# Step 0 — Flying Drones: Applicant Extraction & Name Harmonization

**Contributor:** Soroush  
**Purpose:** Continue the team pipeline *after* Ricky's patent selection and Basar's LLM validation. **No SBERT or LLM is rerun here.**

**Inputs (use one):**
1. `Step0/applicants_raw.csv` if your teammate has already exported applicants; or
2. `df_selected.parquet` exported from `1-DataPreparation.ipynb` (the code currently saves it as `/home/jovyan/progetto/df_selected.parquet`).

**Outputs:** `outputs/applicant_harmonization/applicants_raw.csv`, `applicants_harmonized.csv`, `applicant_entity_mapping.csv`, `family_applicant_links.csv`, `applicant_manual_review.csv`, `summary.csv`.

> **Method:** preserve original applicant records; prefer PATSTAT PSN names/IDs when present; never merge names just because they share the same first word. HAN data is retained if available but not automatically treated as equivalent to legal entity identity. These are provisional organization mappings requiring manual checks for major players. Natural-person applicants may also occur.

> **Important:** this runs on the families supplied in the input. The input is provisional until the team confirms final patent selection. Run it again with the final family set.


## 1. Set paths and choose the existing input

Run this Notebook from the `Step0/` folder (or the repository root). We first look for an already extracted applicant CSV; otherwise use Ricky's selected applications Parquet. No need to rerun data preparation if that file already exists.

In [1]:
from pathlib import Path
import re
import unicodedata
import pandas as pd
from IPython.display import display

HERE = Path.cwd()
STEP0 = HERE if (HERE / "1-DataPreparation.ipynb").exists() else HERE / "Step0"
OUT = STEP0 / "outputs" / "applicant_harmonization"
OUT.mkdir(parents=True, exist_ok=True)

RAW_CANDIDATES = [STEP0 / "applicants_raw.csv", STEP0 / "outputs" / "applicants_raw.csv"]
SELECTED_CANDIDATES = [
    STEP0 / "df_selected.parquet",
    HERE / "df_selected.parquet",
    Path("/home/jovyan/progetto/df_selected.parquet"),
]
raw_path = next((p for p in RAW_CANDIDATES if p.exists()), None)
selected_path = next((p for p in SELECTED_CANDIDATES if p.exists()), None)
print("Current working directory:", HERE)
print("Step0 folder:", STEP0)
print("Existing applicant CSV:", raw_path)
print("Selected patents Parquet:", selected_path)
if raw_path is None and selected_path is None:
    raise FileNotFoundError(
        "No applicant CSV or df_selected.parquet found. Ask Ricky/Basar for the "
        "existing df_selected.parquet export (do not rerun SBERT). "
        "Place it in Step0/ or keep it at /home/jovyan/progetto/."
    )


Current working directory: /home/jovyan/flying-drones-patent-analytics/Step0
Step0 folder: /home/jovyan/flying-drones-patent-analytics/Step0
Existing applicant CSV: None
Selected patents Parquet: /home/jovyan/progetto/df_selected.parquet


## 2. Extract Applicant records (only if a CSV is not already available)

For the **selected DOCDB families**, join all available family members to `TLS207_PERS_APPLN` and `TLS206_PERSON`. `applt_seq_nr > 0` selects **applicants**, not inventor-only records. Chunking protects against large `IN (...)` clauses. Optional PSN/HAN columns are included when supported by the installed PATSTAT model.

This step **reads patent metadata**, not scientific articles or LLM validation results. It does not alter Basar's existing notebooks.

In [2]:
if raw_path is not None:
    df_applicants = pd.read_csv(raw_path)
    print("Loaded previously extracted applicant data:", raw_path)
else:
    df_selected = pd.read_parquet(selected_path)
    required = {"appln_id", "docdb_family_id"}
    assert required.issubset(df_selected.columns), f"Missing columns: {required - set(df_selected.columns)}"
    family_ids = pd.to_numeric(df_selected["docdb_family_id"], errors="coerce")
    selected_family_ids = sorted(set(family_ids[family_ids.gt(0)].astype("int64")))
    print("Selected DOCDB families:", len(selected_family_ids))
    if not selected_family_ids:
        raise ValueError("No valid selected family IDs")

    from epo.tipdata.patstat import PatstatClient
    from epo.tipdata.patstat.database.models import (
        TLS201_APPLN, TLS206_PERSON, TLS207_PERS_APPLN
    )
    # Match this to the environment used for the selected patent dataset.
    ENV = "TEST"
    patstat = PatstatClient(env=ENV)
    db = patstat.orm()

    cols = [
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_id,
        TLS206_PERSON.person_id,
        TLS206_PERSON.person_name,
        TLS206_PERSON.psn_name,
        TLS206_PERSON.person_ctry_code,
    ]
    for field in ["psn_id", "han_name", "han_id", "doc_std_name"]:
        if hasattr(TLS206_PERSON, field):
            cols.append(getattr(TLS206_PERSON, field))

    pieces = []
    batch_size = 250
    for start in range(0, len(selected_family_ids), batch_size):
        batch = selected_family_ids[start:start+batch_size]
        q = (
            db.query(*cols)
            .select_from(TLS201_APPLN)
            .join(TLS207_PERS_APPLN,
                  TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
            .join(TLS206_PERSON,
                  TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
            .filter(TLS201_APPLN.docdb_family_id.in_(batch))
            .filter(TLS207_PERS_APPLN.applt_seq_nr > 0)
            .distinct()
        )
        part = patstat.df(q)
        if len(part):
            pieces.append(part)
        print(f"Applicant batches: {min(start+batch_size, len(selected_family_ids))}/{len(selected_family_ids)} families")

    if not pieces:
        raise ValueError("PATSTAT returned no applicant records for the selected families")
    df_applicants = pd.concat(pieces, ignore_index=True)

mandatory = {"docdb_family_id", "appln_id", "person_id", "person_name", "psn_name", "person_ctry_code"}
missing = mandatory - set(df_applicants.columns)
if missing:
    raise ValueError(f"Applicant input missing required columns: {sorted(missing)}")

df_applicants = df_applicants.drop_duplicates(subset=["docdb_family_id", "appln_id", "person_id"]).copy()
print("Application-applicant relationships:", len(df_applicants))
print("Distinct applicant person IDs:", df_applicants.person_id.nunique())
print("Distinct families with applicant records:", df_applicants.docdb_family_id.nunique())
display(df_applicants.head())
df_applicants.to_csv(OUT / "applicants_raw.csv", index=False)


Selected DOCDB families: 2892
Applicant batches: 250/2892 families
Applicant batches: 500/2892 families
Applicant batches: 750/2892 families
Applicant batches: 1000/2892 families
Applicant batches: 1250/2892 families
Applicant batches: 1500/2892 families
Applicant batches: 1750/2892 families
Applicant batches: 2000/2892 families
Applicant batches: 2250/2892 families
Applicant batches: 2500/2892 families
Applicant batches: 2750/2892 families
Applicant batches: 2892/2892 families
Application-applicant relationships: 5031
Distinct applicant person IDs: 3217
Distinct families with applicant records: 2872


,docdb_family_id,appln_id,person_id,person_name,psn_name,person_ctry_code,psn_id,han_name,han_id,doc_std_name
0,20407738,51073300,8116198,Forsvarets Materielverk,FOERSVARETS MATERIELVERK,SE,8987796,FORSVARETS MATERIELVERK,1099515,FORSVARETS MATERIELVERK
1,39585122,46407155,5453378,"Intelligent Technologies International, Inc.",INTELLIGENT TECHNOLOGIES INTERNATIONAL,US,14089163,INTELLIGENT TECH INT INC,1587467,BREED DAVID S
2,39585122,46407155,5453379,"Breed, David S.","BREED, DAVID S.",US,3381899,"Breed, David S.",105453379,BREED DAVID S
3,34714741,53594691,5236224,"Statsignal IPC, LLC",STATSIGNAL IPC,US,30944304,STATSIGNAL IPC LLC,2783781,STATSIGNAL IPC LLC
4,34714741,48267780,5457789,"Sipco, LLC",SIPCO,US,30277990,SIPCO LLC,2707732,SIPCO LLC


## 3. Conservative name harmonization

Normalize case/Unicode/spacing and prefer `psn_name` over `person_name`. For entity IDs, prefer **positive `psn_id`**, if available; otherwise use the original `person_id`. This is intentionally conservative: it avoids automatically merging two distinct entities simply because their spellings are similar.

The **HAN fields are retained for later corporate-group mapping**, not automatically used to collapse distinct legal entities.

In [4]:
import re
import pandas as pd
from IPython.display import display
from cleanco import basename

# Choose the method used for the final display name:
# "psn", "dictionary", or "cleanco"
NAME_METHOD = "psn"

# Professor's legal-form dictionary
LEGAL_FORMS = {
    "INC", "INCORPORATED", "CORP", "CORPORATION",
    "CO", "COMPANY", "LTD", "LIMITED", "LLC",
    "LP", "PLC", "GMBH", "AG", "KG", "MBH",
    "SE", "SA", "SAS", "SPA", "SRL", "BV",
    "NV", "AB", "AS", "ASA", "OY", "OYJ",
    "KK", "KABUSHIKI", "KAISHA", "LTDA",
    "PTY", "THE", "AKTIENGESELLSCHAFT",
    "GESELLSCHAFT"
}

# Basic normalization
def clean_names(series):
    return (
        series.astype("string")
        .str.normalize("NFKC")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.upper()
        .replace("", pd.NA)
    )

# METHOD A — Professor's Dictionary Cleaning
def clean_with_dictionary(name):
    if pd.isna(name) or not str(name).strip():
        return pd.NA

    words = re.sub(
        r"[^A-Z0-9 ]", " ", str(name).upper()
    ).split()

    while words and words[-1] in LEGAL_FORMS:
        words.pop()

    while words and words[0] in LEGAL_FORMS:
        words.pop(0)

    return " ".join(words) if words else pd.NA

# METHOD B — Professor's Cleanco
def clean_with_cleanco(name):
    if pd.isna(name) or not str(name).strip():
        return pd.NA

    return clean_with_dictionary(
        basename(str(name))
    )

# METHOD C — PATSTAT PSN baseline
# Use our actual Flying Drones applicants
h = df_applicants.copy()

h["person_name_clean"] = clean_names(
    h["person_name"]
)
h["psn_name_clean"] = clean_names(
    h["psn_name"]
)

base_names = h["psn_name_clean"].fillna(
    h["person_name_clean"]
)

h["name_source"] = "person_name"
h.loc[
    h["psn_name_clean"].notna(),
    "name_source"
] = "psn_name"

# Calculate all three methods
h["candidate_psn"] = base_names

h["candidate_dictionary"] = (
    base_names.map(clean_with_dictionary)
    .astype("string")
)

h["candidate_cleanco"] = (
    base_names.map(clean_with_cleanco)
    .astype("string")
)

methods = {
    "psn": "candidate_psn",
    "dictionary": "candidate_dictionary",
    "cleanco": "candidate_cleanco"
}

h["standardized_name"] = (
    h[methods[NAME_METHOD]].fillna(base_names)
)

# Keep PATSTAT identity identifiers conservative.
# Other name-cleaning methods are candidates for review,
# NOT automatically confirmed corporate identities.
person_ids = pd.to_numeric(
    h["person_id"], errors="coerce"
)

if person_ids.isna().any():
    raise ValueError("Missing person_id")

h["entity_id"] = (
    "PERSON:" +
    person_ids.astype("int64").astype(str)
)
h["identity_rule"] = "original_person_id"

if "psn_id" in h.columns:
    psn_ids = pd.to_numeric(
        h["psn_id"], errors="coerce"
    )
    valid = psn_ids.gt(0)

    h.loc[valid, "entity_id"] = (
        "PSN:" +
        psn_ids.loc[valid].astype("int64").astype(str)
    )
    h.loc[valid, "identity_rule"] = "patstat_psn_id"

# Data integrity checks
assert len(h) == len(df_applicants)
assert not h[
    ["docdb_family_id", "appln_id", "person_id"]
].duplicated().any()

# Compare methods
comparison = pd.DataFrame({
    "Method": [
        "Original person names",
        "PATSTAT PSN",
        "Dictionary",
        "Cleanco",
        "Conservative entity IDs"
    ],
    "Unique values": [
        h["person_name_clean"].nunique(),
        h["candidate_psn"].nunique(),
        h["candidate_dictionary"].nunique(),
        h["candidate_cleanco"].nunique(),
        h["entity_id"].nunique()
    ]
})

print("APPLICATION–APPLICANT RECORDS:", len(h))
print("SELECTED METHOD:", NAME_METHOD)
print("\nHARMONIZATION COMPARISON")
display(comparison)

# Inspect the actual transformations
display(
    h[[
        "person_name",
        "psn_name",
        "candidate_dictionary",
        "candidate_cleanco",
        "standardized_name",
        "entity_id"
    ]].head(20)
)

# Potential alias groups: different PATSTAT entities
# that become the same dictionary-cleaned name.
alias_candidates = (
    h.dropna(subset=["candidate_dictionary"])
    .groupby(
        ["candidate_dictionary", "person_ctry_code"],
        dropna=False
    )
    .agg(
        distinct_entities=("entity_id", "nunique"),
        example_names=(
            "person_name",
            lambda s: " | ".join(
                sorted(set(s.dropna().astype(str)))[:5]
            )
        )
    )
    .reset_index()
)

alias_candidates = alias_candidates[
    alias_candidates["distinct_entities"] > 1
].sort_values(
    "distinct_entities", ascending=False
)

print("\nPOTENTIAL ALIAS GROUPS:", len(alias_candidates))
display(alias_candidates.head(20))

alias_candidates.to_csv(
    OUT / "dictionary_alias_candidates.csv",
    index=False
)


APPLICATION–APPLICANT RECORDS: 5031
SELECTED METHOD: psn

HARMONIZATION COMPARISON


,Method,Unique values
0,Original person names,2784
1,PATSTAT PSN,2625
2,Dictionary,2581
3,Cleanco,2579
4,Conservative entity IDs,2640


,person_name,psn_name,candidate_dictionary,candidate_cleanco,standardized_name,entity_id
0,Forsvarets Materielverk,FOERSVARETS MATERIELVERK,FOERSVARETS MATERIELVERK,FOERSVARETS MATERIELVERK,FOERSVARETS MATERIELVERK,PSN:8987796
1,"Intelligent Technologies International, Inc.",INTELLIGENT TECHNOLOGIES INTERNATIONAL,INTELLIGENT TECHNOLOGIES INTERNATIONAL,INTELLIGENT TECHNOLOGIES INTERNATIONAL,INTELLIGENT TECHNOLOGIES INTERNATIONAL,PSN:14089163
2,"Breed, David S.","BREED, DAVID S.",BREED DAVID S,BREED DAVID S,"BREED, DAVID S.",PSN:3381899
3,"Statsignal IPC, LLC",STATSIGNAL IPC,STATSIGNAL IPC,STATSIGNAL IPC,STATSIGNAL IPC,PSN:30944304
4,"Sipco, LLC",SIPCO,SIPCO,SIPCO,SIPCO,PSN:30277990
5,"Petite, Thomas David","PETITE, THOMAS DAVID",PETITE THOMAS DAVID,PETITE THOMAS DAVID,"PETITE, THOMAS DAVID",PSN:25453024
6,Guice; David L.,GUICE; DAVID L.,GUICE DAVID L,GUICE DAVID L,GUICE; DAVID L.,PSN:11219631
7,"Dent, Jr.; William V.","DENT, JR.; WILLIAM V.",DENT JR WILLIAM V,DENT JR WILLIAM V,"DENT, JR.; WILLIAM V.",PSN:6619532
8,Green; Augustus H.,GREEN; AUGUSTUS H.,GREEN AUGUSTUS H,GREEN AUGUSTUS H,GREEN; AUGUSTUS H.,PSN:10871647
9,EUROCOPTER,EUROCOPTER,EUROCOPTER,EUROCOPTER,EUROCOPTER,PSN:8335213



POTENTIAL ALIAS GROUPS: 28


,candidate_dictionary,person_ctry_code,distinct_entities,example_names
2689,ZSM HOLDINGS,US,4,ZSM HOLDINGS LLC | ZSM Holdings LLC
2294,TOP SEVEN,DE,3,TOP SEVEN GMBH & CO. KG | TOP seven GmbH & Co. KG
2473,WORKHORSE GROUP,US,3,Workhorse Group Inc | Workhorse Group Inc.
19,ACSL,JP,2,ACSL LTD. | ACSL Ltd.
435,CIRRUS DESIGN CORPORATION D B A CIRRUS AIRCRAFT,US,2,CIRRUS DESIGN CORPORATION D/B/A CIRRUS AIRCRAF...
478,COWDEN TECHNOLOGIES,US,2,"COWDEN TECHNOLOGIES, INC. | Cowden Technologie..."
741,FUKUSHIMA SANGIKYO,JP,2,FUKUSHIMA SANGIKYO CORPORATION | Fukushima San...
907,HAN KYUNG HEE,KR,2,"HAN, KYUNG HEE | Han, Kyung Hee"
1018,HOOD TECHNOLOGY,US,2,HOOD TECHNOLOGY CORPORATION | Hood Technology ...
74,ALEXCO HOLDING,DE,2,AlexCo Holding GmbH


## 4. Flag possible aliases for manual review

Two different entity IDs with the same normalized name and country are **candidates**, not proof of identity. They are exported for review but are **not merged automatically**. This is where you can add carefully verified aliases and documented subsidiary rules as a later improvement.

In [5]:

# STEP 0 — CELL 4: FLAG POSSIBLE APPLICANT ALIASES
# Based on the professor's name harmonization methods.
# Potential matches are flagged, not automatically merged.

import pandas as pd
from IPython.display import display

# Preserve the original PSN-based key for compatibility
# with the following export cell.
h["name_country_key"] = (
    h["standardized_name"].fillna("").astype(str)
    + " | "
    + h["person_ctry_code"].fillna("").astype(str).str.upper()
)

def flag_possible_aliases(df, name_column):
    """
    Flag records where the same cleaned applicant name
    and country correspond to different entity IDs.

    This is evidence of a possible alias, not proof
    that the organizations are identical.
    """
    names = df[name_column].astype("string").str.strip()

    countries = (
        df["person_ctry_code"]
        .astype("string")
        .fillna("")
        .str.strip()
        .str.upper()
    )

    temp = pd.DataFrame({
        "name": names,
        "country": countries,
        "entity_id": df["entity_id"]
    }, index=df.index)

    valid = temp["name"].notna() & temp["name"].ne("")

    distinct_ids = (
        temp.loc[valid]
        .groupby(["name", "country"], dropna=False)["entity_id"]
        .transform("nunique")
    )

    flags = pd.Series(False, index=df.index)
    flags.loc[valid] = distinct_ids.gt(1).to_numpy()

    return flags


# Compare possible aliases under all three methods.
h["flag_psn"] = flag_possible_aliases(
    h, "candidate_psn"
)

h["flag_dictionary"] = flag_possible_aliases(
    h, "candidate_dictionary"
)

h["flag_cleanco"] = flag_possible_aliases(
    h, "candidate_cleanco"
)

h["flag_missing_name"] = h["standardized_name"].isna()

# A record needs review if any method raises a flag.
flag_columns = [
    "flag_psn",
    "flag_dictionary",
    "flag_cleanco",
    "flag_missing_name"
]

h["needs_manual_review"] = h[flag_columns].any(axis=1)

# Create a separate manual-review dataset.
review_columns = [
    "docdb_family_id",
    "appln_id",
    "person_id",
    "person_name",
    "psn_name",
    "candidate_dictionary",
    "candidate_cleanco",
    "standardized_name",
    "person_ctry_code",
    "entity_id",
    "identity_rule",
    "name_country_key"
] + flag_columns

review = h.loc[
    h["needs_manual_review"], review_columns
].copy()

review["human_decision"] = ""
review["notes"] = ""

# Summarize the results.
summary_flags = pd.DataFrame({
    "Review condition": [
        "PSN name collision",
        "Dictionary candidate collision",
        "Cleanco candidate collision",
        "Missing standardized name",
        "Total records requiring review"
    ],
    "Number of records": [
        int(h["flag_psn"].sum()),
        int(h["flag_dictionary"].sum()),
        int(h["flag_cleanco"].sum()),
        int(h["flag_missing_name"].sum()),
        len(review)
    ]
})

print("Total applicant records:", len(h))
print("Records requiring manual review:", len(review))
print("Distinct entity IDs flagged:",
      review["entity_id"].nunique())

display(summary_flags)

print("\nEXAMPLES REQUIRING REVIEW")
display(review.head(20))


Total applicant records: 5031
Records requiring manual review: 356
Distinct entity IDs flagged: 62


,Review condition,Number of records
0,PSN name collision,62
1,Dictionary candidate collision,323
2,Cleanco candidate collision,356
3,Missing standardized name,0
4,Total records requiring review,356



EXAMPLES REQUIRING REVIEW


,docdb_family_id,appln_id,person_id,person_name,psn_name,candidate_dictionary,candidate_cleanco,standardized_name,person_ctry_code,entity_id,identity_rule,name_country_key,flag_psn,flag_dictionary,flag_cleanco,flag_missing_name,human_decision,notes
39,21997609,46508399,5246937,The Boeing Company,BOEING COMPANY,BOEING,BOEING,BOEING COMPANY,US,PSN:3019507,patstat_psn_id,BOEING COMPANY | US,False,True,True,False,,
59,22564011,48249502,5230563,The Boeing Company,BOEING COMPANY,BOEING,BOEING,BOEING COMPANY,US,PSN:3019507,patstat_psn_id,BOEING COMPANY | US,False,True,True,False,,
61,30118306,50828059,5246937,The Boeing Company,BOEING COMPANY,BOEING,BOEING,BOEING COMPANY,US,PSN:3019507,patstat_psn_id,BOEING COMPANY | US,False,True,True,False,,
75,32176599,52793483,5210107,The Boeing Company,BOEING COMPANY,BOEING,BOEING,BOEING COMPANY,US,PSN:3019507,patstat_psn_id,BOEING COMPANY | US,False,True,True,False,,
76,32176599,49430546,5210107,The Boeing Company,BOEING COMPANY,BOEING,BOEING,BOEING COMPANY,US,PSN:3019507,patstat_psn_id,BOEING COMPANY | US,False,True,True,False,,
77,32176599,52808632,5210107,The Boeing Company,BOEING COMPANY,BOEING,BOEING,BOEING COMPANY,US,PSN:3019507,patstat_psn_id,BOEING COMPANY | US,False,True,True,False,,
114,34961735,53463496,5210107,The Boeing Company,BOEING COMPANY,BOEING,BOEING,BOEING COMPANY,US,PSN:3019507,patstat_psn_id,BOEING COMPANY | US,False,True,True,False,,
115,36001269,54078762,6548211,THE BOEING COMPANY,BOEING COMPANY,BOEING,BOEING,BOEING COMPANY,US,PSN:3019507,patstat_psn_id,BOEING COMPANY | US,False,True,True,False,,
124,36574986,45460189,5210107,The Boeing Company,BOEING COMPANY,BOEING,BOEING,BOEING COMPANY,US,PSN:3019507,patstat_psn_id,BOEING COMPANY | US,False,True,True,False,,
158,38321107,47762051,5210107,The Boeing Company,BOEING COMPANY,BOEING,BOEING,BOEING COMPANY,US,PSN:3019507,patstat_psn_id,BOEING COMPANY | US,False,True,True,False,,


In [6]:
# One row per distinct entity within each Cleanco candidate group
unique_review = (
    h.loc[
        h["needs_manual_review"],
        [
            "candidate_cleanco",
            "person_ctry_code",
            "entity_id",
            "psn_name",
            "person_name",
            "han_name"
        ]
    ]
    .drop_duplicates(
        subset=[
            "candidate_cleanco",
            "person_ctry_code",
            "entity_id"
        ]
    )
    .sort_values([
        "candidate_cleanco",
        "person_ctry_code",
        "entity_id"
    ])
    .reset_index(drop=True)
)

print("Unique entity-level review entries:", len(unique_review))
display(unique_review.head(40))

Unique entity-level review entries: 62


,candidate_cleanco,person_ctry_code,entity_id,psn_name,person_name,han_name
0,ACSL,JP,PSN:183638728,ACSL Ltd.,ACSL Ltd.,ACSL LTD
1,ACSL,JP,PSN:184923856,ACSL LTD.,ACSL LTD.,ACSL LTD
2,ALEXCO HOLDING,DE,PSN:185262083,AlexCo Holding GmbH,AlexCo Holding GmbH,ALEXCO HOLDING GMBH
3,ALEXCO HOLDING,DE,PSN:681000,ALEXCO HOLDING,AlexCo Holding GmbH,ALEXCO HOLDING GMBH
4,AMAZING SUN,KR,PSN:185834888,"Amazing Sun Co., Ltd.","Amazing Sun Co., Ltd.",AMAZING SUN CO LTD
5,AMAZING SUN,KR,PSN:852792,AMAZING SUN COMPANY,"AMAZING SUN CO., LTD.",AMAZING SUN CO LTD
6,AMPYX POWER,NL,PSN:182955448,Ampyx Power B.V.,Ampyx Power B.V.,AMPYX POWER BV
7,AMPYX POWER,NL,PSN:893114,AMPYX POWER,AMPYX POWER B.V.,AMPYX POWER BV
8,BOEING,US,PSN:189177142,The Boeing Company,The Boeing Company,THE BOEING CO
9,BOEING,US,PSN:3019507,BOEING COMPANY,The Boeing Company,THE BOEING CO


In [7]:

# STEP 0 — APPLICANT HARMONIZATION
# Cell 4B: Prepare alias groups for manual review
#
# Based on professor's Dictionary / Cleanco methodology.
# This cell DOES NOT merge applicants or modify PATSTAT IDs.

from pathlib import Path
import pandas as pd
from IPython.display import display

OUT = Path(OUT)
OUT.mkdir(parents=True, exist_ok=True)

# 1. Prepare the existing harmonized applicant records
candidates = h.copy()

candidates["_candidate_name"] = (
    candidates["candidate_cleanco"]
    .astype("string")
    .str.strip()
)

candidates["_country"] = (
    candidates["person_ctry_code"]
    .astype("string")
    .fillna("")
    .str.strip()
    .str.upper()
)

candidates["_entity"] = (
    candidates["entity_id"]
    .astype("string")
    .str.strip()
)

candidates = candidates.dropna(
    subset=["_candidate_name", "_entity"]
)

candidates = candidates[
    candidates["_candidate_name"].ne("")
    & candidates["_entity"].ne("")
].copy()


# 2. Helper: collect unique names without repeating them
def unique_values(series, limit=15):
    values = sorted({
        str(value).strip()
        for value in series.dropna()
        if str(value).strip()
    })
    result = " | ".join(values[:limit])

    if len(values) > limit:
        result += f" | ... (+{len(values) - limit} more)"

    return result


# 3. Build one review row per candidate group
review_rows = []

for (name, country), group in candidates.groupby(
    ["_candidate_name", "_country"],
    sort=True
):
    entity_ids = sorted(group["_entity"].unique())

    # Only review groups containing different identities
    if len(entity_ids) < 2:
        continue

    # Details for every entity in the group
    entity_details = []

    for entity_id, entity_rows in group.groupby("_entity"):
        families = entity_rows["docdb_family_id"].nunique()
        original_names = unique_values(
            entity_rows["person_name"], limit=3
        )

        entity_details.append(
            f"{entity_id} ({families} families): "
            f"{original_names}"
        )

    review_rows.append({
        "candidate_name": name,
        "country": country,
        "n_entities": len(entity_ids),
        "n_families": group["docdb_family_id"].nunique(),
        "n_records": len(group),
        "entity_ids": " | ".join(entity_ids),
        "psn_names": unique_values(group["psn_name"]),
        "original_names": unique_values(
            group["person_name"]
        ),
        "han_names": (
            unique_values(group["han_name"])
            if "han_name" in group.columns else ""
        ),
        "han_ids": (
            unique_values(group["han_id"])
            if "han_id" in group.columns else ""
        ),
        "entity_details": " ; ".join(entity_details),

        # To be completed after reviewing the evidence
        "decision": "",
        "canonical_entity_id": "",
        "review_notes": ""
    })


# 4. Prioritize groups affecting more patent families
alias_review_groups = pd.DataFrame(review_rows)

if not alias_review_groups.empty:
    alias_review_groups = (
        alias_review_groups
        .sort_values(
            ["n_families", "n_entities", "candidate_name"],
            ascending=[False, False, True]
        )
        .reset_index(drop=True)
    )

    alias_review_groups.insert(
        0,
        "group_id",
        range(1, len(alias_review_groups) + 1)
    )

# 5. Export review table
output_file = OUT / "alias_review_groups.csv"

alias_review_groups.to_csv(
    output_file,
    index=False
)

print("APPLICANT ALIAS REVIEW")
print("----------------------")
print("Candidate groups:", len(alias_review_groups))

if not alias_review_groups.empty:
    print(
        "Distinct entity IDs:",
        len({
            entity
            for value in alias_review_groups["entity_ids"]
            for entity in value.split(" | ")
        })
    )

    print(
        "Patent families affected:",
        candidates.loc[
            candidates.set_index(
                ["_candidate_name", "_country"]
            ).index.isin(
                alias_review_groups.set_index(
                    ["candidate_name", "country"]
                ).index
            ),
            "docdb_family_id"
        ].nunique()
    )

print("\nSaved to:", output_file.resolve())

display(
    alias_review_groups[
        [
            "group_id",
            "candidate_name",
            "country",
            "n_entities",
            "n_families",
            "entity_ids",
            "psn_names",
            "han_names",
            "decision"
        ]
    ].head(30)
)


APPLICANT ALIAS REVIEW
----------------------
Candidate groups: 29
Distinct entity IDs: 62
Patent families affected: 173

Saved to: /home/jovyan/flying-drones-patent-analytics/Step0/outputs/applicant_harmonization/alias_review_groups.csv


,group_id,candidate_name,country,n_entities,n_families,entity_ids,psn_names,han_names,decision
0,1,BOEING,US,2,103,PSN:189177142 | PSN:3019507,BOEING COMPANY | The Boeing Company,THE BOEING CO,
1,2,NEARTHLAB,KR,2,13,PSN:189236384 | PSN:23284374,NEARTHLAB | NEARTHLAB INC.,NEARTHLAB INC,
2,3,LOVELAND INNOVATIONS,US,2,11,PSN:183543266 | PSN:19991998,"LOVELAND INNOVATIONS | Loveland Innovations, Inc.",LOVELAND INNOVATIONS INC | LOVELAND INNOVATION...,
3,4,ZSM HOLDINGS,US,4,6,PSN:181796849 | PSN:183390155 | PSN:185446973 ...,ZSM HOLDINGS | ZSM HOLDINGS LLC | ZSM Holdings...,ZSM HOLDINGS LLC,
4,5,HOOD TECHNOLOGY,US,2,5,PSN:13131154 | PSN:182966898,HOOD TECHNOLOGY CORPORATION | Hood Technology ...,HOOD TECH CORP,
5,6,AMPYX POWER,NL,2,4,PSN:182955448 | PSN:893114,AMPYX POWER | Ampyx Power B.V.,AMPYX POWER BV,
6,7,ACSL,JP,2,3,PSN:183638728 | PSN:184923856,ACSL LTD. | ACSL Ltd.,ACSL LTD,
7,8,TOP SEVEN,DE,3,2,PSN:184618437 | PSN:185997099 | PSN:32972213,TOP SEVEN & COMPANY | TOP SEVEN GMBH & CO. KG,TOP SEVEN GMBH & CO KG,
8,9,HOVERFLY TECHNOLOGIES,US,2,2,PSN:13270048 | PSN:190874678,HOVERFLY TECHNOLOGIES | Hoverfly Technologies....,HOVERFLY TECH INC | Hoverfly Technologies. Inc.,
9,10,PERCEPTUAL ROBOTICS,GB,2,2,PSN:183464445 | PSN:25296726,PERCEPTUAL ROBOTICS | PERCEPTUAL ROBOTICS LIMITED,PERCEPTUAL ROBOTICS LTD,


## 5. Export the four deliverables and quality checks

Save a traceable mapping and a **deduplicated family-to-applicant link table** for downstream competitive-landscape analysis. Counting applications or CSV rows as patents would overcount the inventions; use `docdb_family_id` at the family level.

In [ ]:
h.drop(columns=["name_country_key"]).to_csv(OUT / "applicants_harmonized.csv", index=False)

mapping_cols = [c for c in [
    "person_id", "person_name", "psn_name", "psn_id", "han_name", "han_id",
    "person_ctry_code", "standardized_name", "entity_id", "identity_rule",
    "needs_manual_review"
] if c in h.columns]
entity_mapping = h[mapping_cols].drop_duplicates().sort_values("entity_id")
entity_mapping.to_csv(OUT / "applicant_entity_mapping.csv", index=False)

family_links = h[["docdb_family_id", "entity_id"]].drop_duplicates()
family_links.to_csv(OUT / "family_applicant_links.csv", index=False)
review.to_csv(OUT / "applicant_manual_review.csv", index=False)

summary = pd.DataFrame([{
    "applicant_records":len(h),
    "families_with_applicants":h.docdb_family_id.nunique(),
    "distinct_person_ids":h.person_id.nunique(),
    "harmonized_entity_ids":h.entity_id.nunique(),
    "manual_review_records":len(review),
}])
summary.to_csv(OUT / "summary.csv", index=False)
display(summary)
print("Outputs saved to:", OUT.resolve())


### Applicant Harmonization — Preliminary Results

We adapted the applicant name harmonization methodology introduced in Tutorial 1, Section 2, to the Flying Drones patent dataset.

Using PATSTAT, we extracted 5,031 application–applicant relationships covering 2,872 DOCDB patent families.

Three naming methods were compared:
- PATSTAT Standardized Names (PSN)
- Dictionary-based legal-form cleaning
- Cleanco-based cleaning

We retained PATSTAT identifiers as the conservative entity baseline and identified 29 potential alias groups involving 62 distinct entity IDs.

**Current status:** Candidate identification is complete. Manual identity verification and approved entity consolidation are pending. No unverified applicant merges have been applied.

The results are preliminary and based on the current TEST patent selection.